# Dispenser 動作確認ノートブック

ディスペンサー（eco-PEN450 / Pololu Tic T500）の実機コマンドを確認します。

**前提条件:**
- ディスペンサーが RPi の GPIO14/15 (UART) に接続済み
- RPi 上で ser2net が起動済み

接続先 IP・ポートは `devices/devices.settings.yaml` の `high_viscosity_dispenser.host` / `ser2net_port` で管理します。  
各セルがそれを自動で読み込むため、IP を変更する場合は YAML だけ編集すれば OK です。


In [52]:
import sys
import os
import importlib.util
from pathlib import Path

# プロジェクトルートの絶対パス
PROJECT_ROOT = Path(os.path.abspath(".."))

def import_device(filename: str):
    """devices/ ディレクトリのファイルを直接インポートする（__init__.py をスキップ）。"""
    path = PROJECT_ROOT / "devices" / filename
    module_name = path.stem
    spec = importlib.util.spec_from_file_location(module_name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_mod_real = import_device("high_viscosity_dispenser_proprietary.py")
HighViscosityDispenserProprietary = _mod_real.HighViscosityDispenserProprietary

print("インポート成功")


インポート成功


---
## コマンド単体テスト（実機）

`HighViscosityDispenserProprietary` を直接インスタンス化し、実機に対してコマンドを確認します。

**前提条件:**
- ディスペンサーが RPi の GPIO14/15 (UART) に接続済み
- RPi 上で ser2net が起動していること（下記参照）
- Tic Control Center で以下を設定済み:
  - Control mode: **Serial / I2C / USB**
  - Command timeout: **disabled**
  - Baud rate: **9600**
  - Step mode: `microstep_multiplier` と一致（デフォルト: 1/8 step）

接続先は `devices/devices.settings.yaml` の `host` / `ser2net_port` から自動で読み込まれます。

**RPi 上での ser2net セットアップ（初回のみ）:**
```bash
# 1. UART 有効化 + GPIO14/15 設定（RPi5 固有の手順）
#    /boot/firmware/config.txt に以下を追記して再起動:
#      dtoverlay=uart0-pi5     # GPIO14/15 を UART0 にマップ（RPi5 専用）
#      dtoverlay=disable-bt    # Bluetooth を GPIO14/15 から切り離す
#    起動時に GPIO14/15 を ALT4(UART) に設定するサービスを追加（uart0-gpio-setup.service）

# 2. ser2net インストール
sudo apt install ser2net

# 3. /etc/ser2net.yaml を編集して以下を追記
#   connection: &ttyAMA0
#     accepter: tcp,2217
#     connector: serialdev,/dev/ttyAMA0,9600n81,local
#     options:
#       kickolduser: true

# 4. ser2net を自動起動設定 + 開始
sudo systemctl enable --now ser2net
```

> **注意**: 実機を動かすため、ノズル先端・吐出先の安全を確認してから実行してください。


In [50]:
# ---- 3-2. 実機の初期化 ----
# 接続先は devices/devices.settings.yaml から自動で読み込まれます。
# host が設定されている場合は ser2net (raw TCP) 経由、null の場合はローカルポート直接接続。
import yaml

_devices_cfg = yaml.safe_load(
    (PROJECT_ROOT / "devices" / "devices.settings.yaml").read_text(encoding="utf-8")
)
_dcfg = _devices_cfg["high_viscosity_dispenser"]

# ↓ 実機で確認した安全な purge 速度 [rev/s] を設定（最大 2.0）
PURGE_SPEED_RPS = 0.5  # 0.5 rev/s = 1.5 mL/min

dispenser_real = HighViscosityDispenserProprietary(
    host=_dcfg.get("host"),                        # Tailscale IP（None = ローカル直接接続）
    ser2net_port=_dcfg.get("ser2net_port", 2217),
    port=_dcfg["port"],
    full_steps_per_rev=_dcfg["full_steps_per_rev"],
    microstep_multiplier=_dcfg["microstep_multiplier"],
    purge_speed_rps=PURGE_SPEED_RPS,
    baud_rate=_dcfg["baud_rate"],
)
print(f"status: {dispenser_real.status}")

status: connected


In [51]:
# ---- 3-3. purge（パージ）----
# ノズル先端が容器の上にあることを確認してから実行してください。

PURGE_VOLUME_ML = 0.1  # パージ量 [mL]

print(f"purge: {PURGE_VOLUME_ML} mL ...")
dispenser_real.purge(volume_ml=PURGE_VOLUME_ML)
print("purge 完了")

purge: 0.1 mL ...
purge 完了


In [ ]:
# ---- 3-4. dispense（吐出） ----
DISPENSE_VOLUME_ML = 0.05
DISPENSE_SPEED_ML_PER_MIN = 2.0

print(f"dispense: {DISPENSE_VOLUME_ML} mL @ {DISPENSE_SPEED_ML_PER_MIN} mL/min ...")
dispenser_real.dispense(volume_ml=DISPENSE_VOLUME_ML, speed_ml_per_min=DISPENSE_SPEED_ML_PER_MIN)
print("dispense 完了")

In [ ]:
# ---- 3-5. suck_back（サックバック） ----
SUCK_BACK_DELAY_S = 0.5
SUCK_BACK_VOLUME_ML = 0.01
SUCK_BACK_SPEED_ML_PER_MIN = 2.0

print(f"suck_back: delay={SUCK_BACK_DELAY_S}s, {SUCK_BACK_VOLUME_ML} mL @ {SUCK_BACK_SPEED_ML_PER_MIN} mL/min ...")
dispenser_real.suck_back(
    volume_ml=SUCK_BACK_VOLUME_ML,
    speed_ml_per_min=SUCK_BACK_SPEED_ML_PER_MIN,
    delay_s=SUCK_BACK_DELAY_S,
)
print("suck_back 完了")

In [48]:
# ---- 3-6. start_rotation / stop_rotation（連続回転）----
# 連続的に回転を開始し、手動で stop_rotation() を呼ぶまで回り続ける。
# 流量の目視確認や、長時間連続吐出のテストに使用。
import time

START_SPEED_RPS = 0.5   # 回転速度 [rev/s]（0.5 = 1.5 mL/min）
DIRECTION = +1          # +1: 吐出方向, -1: サックバック方向
RUN_DURATION_S = 3.0    # 連続回転時間 [s]（テスト用）

print(f"start_rotation: {START_SPEED_RPS} rev/s, direction={DIRECTION}")
dispenser_real.start_rotation(speed_rps=START_SPEED_RPS, direction=DIRECTION)
time.sleep(RUN_DURATION_S)
dispenser_real.stop_rotation()
print(f"{RUN_DURATION_S} 秒後に停止完了")

start_rotation: 0.5 rev/s, direction=1
3.0 秒後に停止完了


In [35]:
# ---- 3-7. check_status ----
dispenser_real.check_status()
print(f"status: {dispenser_real.status}")

status: connected


In [ ]:
# ---- 3-8. 切断 ----
# モーターを安全停止してシリアル接続を閉じる。
# テスト完了後は必ず実行してください。

dispenser_real.close()
print(f"status: {dispenser_real.status}")